<a href="https://colab.research.google.com/github/akul-bharadwaj/various-agents/blob/main/Case_Study_2_Wealth_Management_State_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study 2: Agent Loop with State Management  
## Wealth Management Agent

### Objective

Build a wealth-management agent that collects:

- monthly income;
- monthly expenses; and
- investment goals.

The agent must:

1. persist known inputs in a state object;
2. avoid asking for information already stored;
3. log every state transition;
4. continue until the financial profile is complete; and
5. demonstrate what fails when state is not maintained.

This implementation uses the **OpenAI Responses API with function calling**, but no agent framework such as LangChain, CrewAI, or the OpenAI Agents SDK.

> **Educational disclaimer:** The generated output is a simplified learning example, not personalised financial advice.

## Architecture

```text
User input
    │
    ▼
┌──────────────────────────┐
│ WealthManagementState    │
│ income                   │
│ expenses                 │
│ investment_goals         │
│ status                   │
└─────────────┬────────────┘
              │
              ▼
┌──────────────────────────┐
│ observe()                │
│ Read only the current    │
│ persisted state          │
└─────────────┬────────────┘
              │
              ▼
┌──────────────────────────┐
│ reason()                 │
│ OpenAI chooses the next  │
│ missing field or finish  │
└─────────────┬────────────┘
              │
              ▼
┌──────────────────────────┐
│ act()                    │
│ Ask user / calculate     │
│ financial snapshot       │
└─────────────┬────────────┘
              │
              ▼
┌──────────────────────────┐
│ update_state()           │
│ Persist data and log     │
│ state transition         │
└─────────────┬────────────┘
              │
        Complete?
       No     │     Yes
       └──────┘      ▼
              Final financial profile
```

### Why state matters

Without persistent state, every iteration behaves like a new conversation. The agent forgets previously collected inputs and may repeatedly ask the same question.

## 1. Install the OpenAI package

In [1]:
!pip -q install --upgrade openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.5 MB/s eta 0:00:00


## 2. Configure the OpenAI API key

The API key is entered securely and is not displayed in notebook output.

In [2]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured.")

Enter your OpenAI API key: ··········
OpenAI API key configured.


## 3. Imports and client

The model name is kept in one variable so it can be changed easily if required.

In [3]:
import copy
import json
import logging
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pprint import pprint
from typing import Any, Callable, Dict, List, Optional

from openai import OpenAI

MODEL = "gpt-4o-mini"
client = OpenAI()

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s",
)
logger = logging.getLogger("wealth_management_agent")

print("Client created.")
print("Selected model:", MODEL)

Client created.
Selected model: gpt-4o-mini


## 4. Define the persistent state object

A dataclass makes the state explicit while still allowing conversion to a dictionary.

The state stores:

- collected user inputs;
- the current workflow status;
- the number of completed iterations;
- the final financial snapshot; and
- a full transition history.

In [4]:
@dataclass
class WealthManagementState:
    """Persistent state for the wealth-management workflow."""

    monthly_income: Optional[float] = None
    monthly_expenses: Optional[float] = None
    investment_goals: Optional[str] = None

    status: str = "COLLECTING_INFORMATION"
    iteration: int = 0
    completed: bool = False

    monthly_surplus: Optional[float] = None
    savings_rate_percent: Optional[float] = None
    financial_snapshot: Optional[Dict[str, Any]] = None

    transition_log: List[Dict[str, Any]] = field(
        default_factory=list
    )

    def known_inputs(self) -> Dict[str, Any]:
        """Return only fields that have already been collected."""
        candidate_inputs = {
            "monthly_income": self.monthly_income,
            "monthly_expenses": self.monthly_expenses,
            "investment_goals": self.investment_goals,
        }

        return {
            key: value
            for key, value in candidate_inputs.items()
            if value not in (None, "")
        }

    def missing_inputs(self) -> List[str]:
        """Return required fields that are not yet available."""
        required_inputs = [
            "monthly_income",
            "monthly_expenses",
            "investment_goals",
        ]

        return [
            field_name
            for field_name in required_inputs
            if getattr(self, field_name) in (None, "")
        ]

    def public_view(self) -> Dict[str, Any]:
        """Return a compact state representation for reasoning."""
        return {
            "known_inputs": self.known_inputs(),
            "missing_inputs": self.missing_inputs(),
            "status": self.status,
            "completed": self.completed,
            "iteration": self.iteration,
        }


initial_state = WealthManagementState()
pprint(initial_state.public_view())

{'completed': False,
 'iteration': 0,
 'known_inputs': {},
 'missing_inputs': ['monthly_income', 'monthly_expenses', 'investment_goals'],
 'status': 'COLLECTING_INFORMATION'}


## 5. Define the actions available to the model

OpenAI is allowed to select exactly one of two actions:

- `request_input` when a required field is missing;
- `finalize_profile` when all required fields are present.

The actual state update and numerical calculations remain in Python.

In [5]:
OPENAI_TOOLS = [
    {
        "type": "function",
        "name": "request_input",
        "description": (
            "Request exactly one currently missing wealth-management "
            "field from the user."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "field_name": {
                    "type": "string",
                    "enum": [
                        "monthly_income",
                        "monthly_expenses",
                        "investment_goals",
                    ],
                    "description": (
                        "A field that is currently listed under "
                        "missing_inputs."
                    ),
                },
                "question": {
                    "type": "string",
                    "description": (
                        "A concise and polite question requesting "
                        "that field."
                    ),
                },
            },
            "required": ["field_name", "question"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "finalize_profile",
        "description": (
            "Complete the profile only when missing_inputs is empty."
        ),
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

print("Available actions:")
for tool in OPENAI_TOOLS:
    print("-", tool["name"])

Available actions:
- request_input
- finalize_profile


## 6. Implement the stateful agent

The agent follows:

```text
observe → reason → act → update_state
```

The model receives both `known_inputs` and `missing_inputs`. A validation layer ensures that it cannot request a field already present in state.

In [6]:
class WealthManagementAgent:
    """A stateful agent that collects a basic financial profile."""

    SYSTEM_INSTRUCTIONS = """
You control a wealth-management information-collection workflow.

Choose exactly one function.

Rules:
1. Read known_inputs and missing_inputs carefully.
2. Never request a field that is already present in known_inputs.
3. If missing_inputs is not empty, call request_input for exactly one
   field from missing_inputs.
4. Collect fields in this preferred order when possible:
   monthly_income, monthly_expenses, investment_goals.
5. If missing_inputs is empty, call finalize_profile.
6. Do not invent financial values.
7. Do not provide ordinary text outside the function call.
"""

    def __init__(
        self,
        client: OpenAI,
        state: Optional[WealthManagementState] = None,
        model: str = MODEL,
    ):
        self.client = client
        self.model = model
        self.state = state or WealthManagementState()

    def observe(self) -> Dict[str, Any]:
        """Read the current persisted state."""
        observation = self.state.public_view()

        logger.info(
            "OBSERVE | known=%s | missing=%s",
            list(observation["known_inputs"].keys()),
            observation["missing_inputs"],
        )

        return observation

    def reason(
        self,
        observation: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Use OpenAI to choose the next valid action."""
        response = self.client.responses.create(
            model=self.model,
            instructions=self.SYSTEM_INSTRUCTIONS,
            input=(
                "Current persistent state:\n"
                + json.dumps(observation, indent=2, default=str)
            ),
            tools=OPENAI_TOOLS,
            tool_choice="required",
            parallel_tool_calls=False,
        )

        function_calls = [
            item
            for item in response.output
            if item.type == "function_call"
        ]

        if len(function_calls) != 1:
            raise RuntimeError(
                "Expected exactly one function call, "
                f"received {len(function_calls)}."
            )

        function_call = function_calls[0]

        try:
            arguments = json.loads(function_call.arguments)
        except json.JSONDecodeError as exc:
            raise RuntimeError(
                "The model returned invalid function arguments."
            ) from exc

        action = {
            "name": function_call.name,
            "arguments": arguments,
            "response_id": response.id,
        }

        self._validate_action(action, observation)

        logger.info(
            "REASON | selected_action=%s | arguments=%s",
            action["name"],
            action["arguments"],
        )

        return action

    def act(
        self,
        action: Dict[str, Any],
        input_provider: Callable[[str, str], Any],
    ) -> Dict[str, Any]:
        """Execute the selected action."""
        if action["name"] == "request_input":
            field_name = action["arguments"]["field_name"]
            question = action["arguments"]["question"]

            value = input_provider(field_name, question)
            validated_value = self._validate_input(
                field_name,
                value,
            )

            result = {
                "event": "INPUT_COLLECTED",
                "field_name": field_name,
                "value": validated_value,
                "next_status": "COLLECTING_INFORMATION",
            }

        elif action["name"] == "finalize_profile":
            result = {
                "event": "PROFILE_FINALIZED",
                "snapshot": self._calculate_snapshot(),
                "next_status": "COMPLETED",
            }

        else:
            raise ValueError(
                f"Unsupported action: {action['name']}"
            )

        logger.info(
            "ACT | event=%s",
            result["event"],
        )

        return result

    def update_state(
        self,
        action: Dict[str, Any],
        result: Dict[str, Any],
    ) -> None:
        """Persist action output and log the state transition."""
        state_before = self.state.public_view()

        if result["event"] == "INPUT_COLLECTED":
            setattr(
                self.state,
                result["field_name"],
                result["value"],
            )
            self.state.status = result["next_status"]

        elif result["event"] == "PROFILE_FINALIZED":
            snapshot = result["snapshot"]

            self.state.monthly_surplus = snapshot[
                "monthly_surplus"
            ]
            self.state.savings_rate_percent = snapshot[
                "savings_rate_percent"
            ]
            self.state.financial_snapshot = snapshot
            self.state.status = result["next_status"]
            self.state.completed = True

        else:
            raise ValueError(
                f"Unknown result event: {result['event']}"
            )

        self.state.iteration += 1
        state_after = self.state.public_view()

        transition = {
            "timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "iteration": self.state.iteration,
            "action": action["name"],
            "event": result["event"],
            "state_before": copy.deepcopy(state_before),
            "state_after": copy.deepcopy(state_after),
        }

        self.state.transition_log.append(transition)

        logger.info(
            "UPDATE_STATE | iteration=%d | status=%s | "
            "known=%s | missing=%s",
            self.state.iteration,
            self.state.status,
            list(state_after["known_inputs"].keys()),
            state_after["missing_inputs"],
        )

    def _validate_action(
        self,
        action: Dict[str, Any],
        observation: Dict[str, Any],
    ) -> None:
        """Prevent the model from violating state rules."""
        missing_inputs = observation["missing_inputs"]

        if missing_inputs:
            if action["name"] != "request_input":
                raise RuntimeError(
                    "The profile is incomplete, so the agent "
                    "must request an input."
                )

            selected_field = action["arguments"]["field_name"]

            if selected_field not in missing_inputs:
                raise RuntimeError(
                    "The model requested a known or unsupported "
                    f"field: {selected_field!r}."
                )

        elif action["name"] != "finalize_profile":
            raise RuntimeError(
                "All inputs are known, so the profile must "
                "be finalized."
            )

    @staticmethod
    def _validate_input(
        field_name: str,
        value: Any,
    ) -> Any:
        """Validate and normalize collected values."""
        if field_name in {
            "monthly_income",
            "monthly_expenses",
        }:
            try:
                numeric_value = float(value)
            except (TypeError, ValueError) as exc:
                raise ValueError(
                    f"{field_name} must be numeric."
                ) from exc

            if numeric_value < 0:
                raise ValueError(
                    f"{field_name} cannot be negative."
                )

            if (
                field_name == "monthly_income"
                and numeric_value == 0
            ):
                raise ValueError(
                    "monthly_income must be greater than zero."
                )

            return numeric_value

        if field_name == "investment_goals":
            text_value = str(value).strip()

            if not text_value:
                raise ValueError(
                    "investment_goals cannot be empty."
                )

            return text_value

        raise ValueError(
            f"Unsupported field: {field_name}"
        )

    def _calculate_snapshot(self) -> Dict[str, Any]:
        """Create a deterministic financial snapshot."""
        income = float(self.state.monthly_income)
        expenses = float(self.state.monthly_expenses)
        surplus = income - expenses
        savings_rate = (surplus / income) * 100

        if surplus < 0:
            cash_flow_status = "DEFICIT"
            educational_note = (
                "Monthly expenses exceed income. Review essential "
                "and discretionary spending before investing."
            )
        elif savings_rate < 10:
            cash_flow_status = "LOW_SURPLUS"
            educational_note = (
                "The monthly surplus is positive but limited. "
                "Consider building an emergency buffer first."
            )
        elif savings_rate < 30:
            cash_flow_status = "MODERATE_SURPLUS"
            educational_note = (
                "The profile has a moderate monthly surplus that "
                "could support goal-based planning."
            )
        else:
            cash_flow_status = "STRONG_SURPLUS"
            educational_note = (
                "The profile has a strong monthly surplus. "
                "Investment allocation should still reflect time "
                "horizon and risk tolerance."
            )

        return {
            "monthly_income": round(income, 2),
            "monthly_expenses": round(expenses, 2),
            "monthly_surplus": round(surplus, 2),
            "savings_rate_percent": round(
                savings_rate,
                2,
            ),
            "cash_flow_status": cash_flow_status,
            "investment_goals": self.state.investment_goals,
            "educational_note": educational_note,
            "disclaimer": (
                "Educational example only; not personalised "
                "financial advice."
            ),
        }


print("WealthManagementAgent defined.")

WealthManagementAgent defined.


## 7. Build the stateful agent loop

The same agent and the same state object are reused during every iteration.

This is what allows the agent to remember previous answers.

In [7]:
def run_stateful_agent(
    input_provider: Callable[[str, str], Any],
    initial_state: Optional[
        WealthManagementState
    ] = None,
    model: str = MODEL,
    max_iterations: int = 10,
    verbose: bool = True,
) -> WealthManagementAgent:
    """Run the agent until the financial profile is complete."""
    agent = WealthManagementAgent(
        client=client,
        state=initial_state,
        model=model,
    )

    while not agent.state.completed:
        if agent.state.iteration >= max_iterations:
            raise RuntimeError(
                f"Agent did not complete within "
                f"{max_iterations} iterations."
            )

        observation = agent.observe()
        action = agent.reason(observation)

        if verbose:
            print(
                f"\n{'=' * 18} "
                f"Iteration {agent.state.iteration + 1} "
                f"{'=' * 18}"
            )
            print("Known inputs:")
            pprint(observation["known_inputs"])
            print("Missing inputs:")
            pprint(observation["missing_inputs"])
            print("Selected action:")
            pprint(
                {
                    "name": action["name"],
                    "arguments": action["arguments"],
                }
            )

        result = agent.act(
            action=action,
            input_provider=input_provider,
        )
        agent.update_state(action, result)

    if verbose:
        print("\n" + "=" * 22)
        print("PROFILE COMPLETED")
        print("=" * 22)
        pprint(agent.state.financial_snapshot)

    return agent

## 8. Demonstration A — Simulated user with persistent state

A dictionary simulates the user's responses so that the notebook can run without manual typing.

Notice that each field is requested only once.

In [8]:
simulated_answers = {
    "monthly_income": 120_000,
    "monthly_expenses": 65_000,
    "investment_goals": (
        "Build a retirement corpus and save for a house "
        "down payment over the next seven years."
    ),
}

question_history: List[str] = []


def simulated_input_provider(
    field_name: str,
    question: str,
) -> Any:
    print(f"Agent: {question}")
    question_history.append(field_name)

    answer = simulated_answers[field_name]
    print(f"User: {answer}")

    return answer


stateful_agent = run_stateful_agent(
    input_provider=simulated_input_provider
)

print("\nFields requested in order:")
print(question_history)

print("\nWas any known field re-asked?")
print(len(question_history) != len(set(question_history)))


================== Iteration 1 ==================
Known inputs:
{}
Missing inputs:
['monthly_income', 'monthly_expenses', 'investment_goals']
Selected action:
{'arguments': {'field_name': 'monthly_income',
               'question': 'Could you please provide your monthly income?'},
 'name': 'request_input'}
Agent: Could you please provide your monthly income?
User: 120000

================== Iteration 2 ==================
Known inputs:
{'monthly_income': 120000.0}
Missing inputs:
['monthly_expenses', 'investment_goals']
Selected action:
{'arguments': {'field_name': 'monthly_expenses',
               'question': 'Could you please provide your monthly expenses?'},
 'name': 'request_input'}
Agent: Could you please provide your monthly expenses?
User: 65000

================== Iteration 3 ==================
Known inputs:
{'monthly_expenses': 65000.0, 'monthly_income': 120000.0}
Missing inputs:
['investment_goals']
Selected action:
{'arguments': {'field_name': 'investment_goals',
         

## 9. Inspect state-transition logs

Each log entry captures:

- the iteration;
- the selected action;
- the event;
- state before the update; and
- state after the update.

This creates an auditable trace of how the agent's memory changed.

In [9]:
for transition in stateful_agent.state.transition_log:
    print(
        f"\nIteration {transition['iteration']} | "
        f"{transition['event']}"
    )
    print("Action:", transition["action"])
    print("State before:")
    pprint(transition["state_before"])
    print("State after:")
    pprint(transition["state_after"])
    print("-" * 75)


Iteration 1 | INPUT_COLLECTED
Action: request_input
State before:
{'completed': False,
 'iteration': 0,
 'known_inputs': {},
 'missing_inputs': ['monthly_income', 'monthly_expenses', 'investment_goals'],
 'status': 'COLLECTING_INFORMATION'}
State after:
{'completed': False,
 'iteration': 1,
 'known_inputs': {'monthly_income': 120000.0},
 'missing_inputs': ['monthly_expenses', 'investment_goals'],
 'status': 'COLLECTING_INFORMATION'}
---------------------------------------------------------------------------

Iteration 2 | INPUT_COLLECTED
Action: request_input
State before:
{'completed': False,
 'iteration': 1,
 'known_inputs': {'monthly_income': 120000.0},
 'missing_inputs': ['monthly_expenses', 'investment_goals'],
 'status': 'COLLECTING_INFORMATION'}
State after:
{'completed': False,
 'iteration': 2,
 'known_inputs': {'monthly_expenses': 65000.0, 'monthly_income': 120000.0},
 'missing_inputs': ['investment_goals'],
 'status': 'COLLECTING_INFORMATION'}
-------------------------------

## 10. Demonstration B — Resume from partially known state

This state already contains income. The agent must not ask for income again; it should request only expenses and goals.

In [10]:
partially_known_state = WealthManagementState(
    monthly_income=150_000
)

remaining_answers = {
    "monthly_expenses": 80_000,
    "investment_goals": (
        "Create a long-term education fund."
    ),
}

resumed_question_history: List[str] = []


def resumed_input_provider(
    field_name: str,
    question: str,
) -> Any:
    print(f"Agent: {question}")
    resumed_question_history.append(field_name)

    answer = remaining_answers[field_name]
    print(f"User: {answer}")

    return answer


resumed_agent = run_stateful_agent(
    input_provider=resumed_input_provider,
    initial_state=partially_known_state,
)

print("\nFields requested:")
print(resumed_question_history)

assert "monthly_income" not in resumed_question_history
print(
    "Success: monthly_income was already in state "
    "and was not requested again."
)


================== Iteration 1 ==================
Known inputs:
{'monthly_income': 150000}
Missing inputs:
['monthly_expenses', 'investment_goals']
Selected action:
{'arguments': {'field_name': 'monthly_expenses',
               'question': 'Could you please provide your monthly expenses?'},
 'name': 'request_input'}
Agent: Could you please provide your monthly expenses?
User: 80000

================== Iteration 2 ==================
Known inputs:
{'monthly_expenses': 80000.0, 'monthly_income': 150000}
Missing inputs:
['investment_goals']
Selected action:
{'arguments': {'field_name': 'investment_goals',
               'question': 'Could you please share your investment goals?'},
 'name': 'request_input'}
Agent: Could you please share your investment goals?
User: Create a long-term education fund.

================== Iteration 3 ==================
Known inputs:
{'investment_goals': 'Create a long-term education fund.',
 'monthly_expenses': 80000.0,
 'monthly_income': 150000}
Missing inp

# Failure Demonstration: Agent Without Persistent State

The following intentionally incorrect loop creates a **new empty state during every iteration**.

Even after the user supplies income, the next iteration starts from an empty object. The agent therefore sees `monthly_income` as missing again and repeats the same question.

The demo is capped at three iterations to avoid an infinite loop.

In [11]:
def demonstrate_stateless_failure(
    input_provider: Callable[[str, str], Any],
    model: str = MODEL,
    iterations: int = 3,
) -> List[str]:
    """
    Intentionally incorrect implementation.

    A new agent and empty state are created on every iteration.
    Collected values are discarded.
    """
    requested_fields: List[str] = []

    for iteration in range(1, iterations + 1):
        # BUG: state is recreated and all previous inputs are lost.
        temporary_state = WealthManagementState()
        temporary_agent = WealthManagementAgent(
            client=client,
            state=temporary_state,
            model=model,
        )

        observation = temporary_agent.observe()
        action = temporary_agent.reason(observation)

        print(
            f"\nStateless iteration {iteration}: "
            f"{action['arguments'].get('question', '')}"
        )

        result = temporary_agent.act(
            action=action,
            input_provider=input_provider,
        )

        requested_field = action["arguments"].get(
            "field_name"
        )
        if requested_field:
            requested_fields.append(requested_field)

        # The update is made only to temporary_state.
        temporary_agent.update_state(action, result)

        print(
            "Temporary known inputs after update:",
            temporary_agent.state.known_inputs(),
        )
        print(
            "This temporary state is now discarded."
        )

    return requested_fields


stateless_answers = {
    "monthly_income": 120_000,
    "monthly_expenses": 65_000,
    "investment_goals": "Retirement planning",
}


def stateless_input_provider(
    field_name: str,
    question: str,
) -> Any:
    answer = stateless_answers[field_name]
    print(f"User supplies: {answer}")
    return answer


stateless_requests = demonstrate_stateless_failure(
    input_provider=stateless_input_provider,
    iterations=3,
)

print("\nFields requested by the stateless loop:")
print(stateless_requests)

print(
    "\nRepeated request detected:",
    len(stateless_requests)
    != len(set(stateless_requests)),
)


Stateless iteration 1: Could you please provide your monthly income?
User supplies: 120000
Temporary known inputs after update: {'monthly_income': 120000.0}
This temporary state is now discarded.

Stateless iteration 2: Could you please provide your monthly income?
User supplies: 120000
Temporary known inputs after update: {'monthly_income': 120000.0}
This temporary state is now discarded.

Stateless iteration 3: Could you please provide your monthly income?
User supplies: 120000
Temporary known inputs after update: {'monthly_income': 120000.0}
This temporary state is now discarded.

Fields requested by the stateless loop:
['monthly_income', 'monthly_income', 'monthly_income']

Repeated request detected: True


## Stateful versus stateless behaviour

| Behaviour | Stateful loop | Stateless loop |
|---|---|---|
| Reuses the same state object | Yes | No |
| Remembers previous answers | Yes | No |
| Avoids asking known fields | Yes | No |
| Can progress to completion | Yes | Usually no |
| Produces transition history | Yes | Fragmented and discarded |
| Typical outcome | Complete profile | Repeated questions |

The failure is not caused by the language model alone. It is primarily an orchestration error: the application fails to preserve state between calls.

## 11. Optional interactive provider

Uncomment the example to enter values manually in Colab.

In [12]:
def interactive_input_provider(
    field_name: str,
    question: str,
) -> Any:
    """Collect one requested value from notebook input."""
    raw_value = input(f"{question}\n> ").strip()

    if field_name in {
        "monthly_income",
        "monthly_expenses",
    }:
        return float(raw_value)

    return raw_value


# Uncomment to run an interactive session:
#
# interactive_agent = run_stateful_agent(
#     input_provider=interactive_input_provider
# )
#
# pprint(interactive_agent.state.financial_snapshot)

## 12. Optional: Save and restore state as JSON

Persistence can extend beyond one Python object. A state snapshot may be saved to disk and restored later.

In [13]:
def save_state_to_json(
    state: WealthManagementState,
    file_path: str,
) -> None:
    """Save the complete state object to a JSON file."""
    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(
            asdict(state),
            file,
            indent=2,
            ensure_ascii=False,
        )


def load_state_from_json(
    file_path: str,
) -> WealthManagementState:
    """Restore a state object from a JSON file."""
    with open(file_path, "r", encoding="utf-8") as file:
        stored_data = json.load(file)

    return WealthManagementState(**stored_data)


state_file = "wealth_management_state.json"
save_state_to_json(
    stateful_agent.state,
    state_file,
)

restored_state = load_state_from_json(state_file)

print("Restored known inputs:")
pprint(restored_state.known_inputs())
print("Restored status:", restored_state.status)
print("Restored transitions:", len(restored_state.transition_log))

Restored known inputs:
{'investment_goals': 'Build a retirement corpus and save for a house down '
                     'payment over the next seven years.',
 'monthly_expenses': 65000.0,
 'monthly_income': 120000.0}
Restored status: COMPLETED
Restored transitions: 4


## How the coding requirements are satisfied

| Requirement | Implementation |
|---|---|
| State object persists inputs | `WealthManagementState` dataclass |
| Collects income | `monthly_income` |
| Collects expenses | `monthly_expenses` |
| Collects goals | `investment_goals` |
| Avoids re-asking known values | `known_inputs()`, `missing_inputs()`, and action validation |
| Agent loop | `run_stateful_agent()` |
| OpenAI API integration | `reason()` uses `client.responses.create()` |
| State-transition logging | `transition_log` and Python `logging` |
| Failure without state | `demonstrate_stateless_failure()` |
| No agent framework | Plain Python plus OpenAI SDK |
| Loop safety | `max_iterations` guard |
| Resumability | Partial-state and JSON examples |

## Key learning

A model does not automatically possess reliable application memory across independent API calls. The application must explicitly preserve and resend relevant state.

OpenAI chooses the next action, but Python remains responsible for:

- storing user information;
- validating inputs;
- preventing repeated questions;
- calculating the financial snapshot; and
- recording an auditable transition history.

## References

- OpenAI function-calling guide:  
  https://developers.openai.com/api/docs/guides/function-calling

- OpenAI Responses API reference:  
  https://developers.openai.com/api/reference/resources/responses/methods/create/

- OpenAI developer quickstart:  
  https://developers.openai.com/api/docs/quickstart